# 09 实践记录：第一次三轮微调完成

这次实际执行了微调，模型参数已经更新，并产生了新的权重。

起点是官方通用 YOLOv5s 的 COCO 预训练权重，不是作者现成的口罩权重。当前模型学习的是三类口罩佩戴检测。

本次目标是验证训练、验证、保存和重新加载流程。三轮不足以据此认定项目准确率已经达标。

## 1. 实际训练设置

| 项目 | 设置 |
|---|---|
| 环境与设备 | dl-study，RTX 5060 Laptop GPU |
| 起始权重 | 官方 YOLOv5 v7.0 的 yolov5s.pt |
| 输入尺寸 | 640 |
| 每批图片数 | 4 |
| 训练轮次 | 3 |
| 数据加载进程 | 0 |
| 随机种子 | 42 |
| 训练/验证图片 | 682 / 171 |
| 优化器 | SGD，沿用仓库默认超参数 |
| 冻结 | 无冻结，默认全模型微调 |

日志显示迁移了 343/349 项参数。类别数由通用 80 类变为 3 类，不兼容部分保留新模型初始化值。

三轮训练主体约 2.6 分钟，环境初始化和最后检查另计；这只是本次运行耗时，不保证下一次相同。

## 2. 逐轮结果怎么读

| 轮次 | 定位损失 | 目标存在性损失 | 分类损失 | 验证 mAP@0.5 |
|---|---:|---:|---:|---:|
| 1 | 0.1056 | 0.0582 | 0.0325 | 5.52% |
| 2 | 0.0761 | 0.0529 | 0.0196 | 20.69% |
| 3 | 0.0626 | 0.0462 | 0.0194 | 22.47% |

训练损失总体下降，验证 mAP@0.5 从约 5.5% 上升到约 22.5%，说明流程确实在学习。最后一轮 mAP@0.5:0.95 约 9.0%。这些数值来自 results.csv；训练末尾对保存权重重新验证的显示值可能有小幅差异。

但“未戴口罩”和“佩戴不正确”在评估器选择的工作点上召回率仍为 0。显示为 1 的某些类别 Precision 不能解读为识别率 100%；没有足够检出时，指标的约定与选择工作点会造成这种显示。

因此，本轮主要确认流程成功，后续需要更多训练，并继续观察每类召回和实际误检漏检。

## 3. 保存了什么

文件均位于项目 `runs/train/mask_smoke_3epochs/`：

| 文件 | 用途 |
|---|---|
| weights/best.pt | 按验证指标选出的权重 |
| weights/last.pt | 最后一轮权重 |
| results.csv | 每轮指标 |
| results.png | 训练与验证曲线 |
| opt.yaml、hyp.yaml | 本次参数与超参数 |
| training-summary.json | 来源、数据指纹、逐轮结果和加载检查 |
| verification/validation-sample.jpg | 新权重检测一张验证图片的结果 |

新 best.pt 已通过当前部署适配器的加载和图片推理功能检查。但可视化发现：验证图片 maksssksksss106 输出的一个框落在左上角卡通标识上，真实人脸漏检。预测与真实框的最大 IoU 为 0。这是一次功能检查通过、识别效果失败的具体样例，进一步说明三轮权重尚不适合直接作为最终模型。

作者原始 models/mask_yolov5.pt 的哈希保持不变，网页默认配置未替换。

## 4. 接下来怎样推进

短程训练已经证明数据、CUDA、训练、验证与保存流程可用。接下来进入较充分的训练，并观察稀少类别是否开始被检出。

当前已保存的 best.pt、last.pt 在结束时经过精简，不包含完整优化器状态；如果把它们作为后续训练起点，应当称为从已有权重继续微调，不能误称为完整恢复同一训练进度。

后续训练完成后，选择实际表现合适的权重接入网页，再检查图片、视频与摄像头的检测效果。

本课未启动长程训练，也未自动切换网页默认模型。